# Notebook 06: Hybrid Retrieval and Reranking

**Project:** Advance AI Research Assistant  
**Module:** Information Retrieval and Ranking  
**Notebook:** `06_hybrid_retrieval_and_reranking.ipynb`  
**Status:** In Progress

---

## 1. Overview

This notebook implements the hybrid information retrieval component of the Advance AI Research Assistant.

The system combines semantic search using ChromaDB with keyword-based retrieval using BM25. The results from both retrieval methods are combined through Reciprocal Rank Fusion (RRF) to produce a unified ranking of relevant document passages.

A Cross-Encoder model is subsequently applied to rerank the retrieved passages based on their relevance to the user's query.

This module aims to improve the accuracy and relevance of scientific information retrieval and provides the retrieval foundation for the Retrieval-Augmented Generation (RAG) pipeline.

## 2. Objectives

The main objectives of this notebook are:

- Load the processed scientific document chunks.
- Implement keyword-based retrieval using BM25.
- Connect to the persistent ChromaDB vector database.
- Implement semantic retrieval using text embeddings.
- Combine keyword and semantic search results using Reciprocal Rank Fusion (RRF).
- Apply Cross-Encoder reranking to improve retrieval relevance.
- Test the retrieval pipeline using scientific research questions.
- Preserve source document and page metadata for citation generation.
- Export retrieval evaluation results for further analysis.

## 3. Tools and Technologies

| Component | Technology |
|---|---|
| Programming language | Python 3.11 |
| Keyword retrieval | BM25 (`rank-bm25`) |
| Semantic retrieval | ChromaDB |
| Embedding framework | Sentence Transformers |
| Embedding model | `all-MiniLM-L6-v2` |
| Ranking fusion | Reciprocal Rank Fusion (RRF) |
| Reranking model | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| Numerical processing | NumPy |
| Data analysis | Pandas |
| Deep learning framework | PyTorch |

## 4. Input and Expected Output

**Input:**
- `data/processed/document_chunks.jsonl`
- `data/chroma_db/` (persistent ChromaDB database)

**Expected outputs:**
- BM25 keyword retrieval results.
- ChromaDB semantic retrieval results.
- Combined hybrid retrieval rankings.
- Cross-Encoder reranked passages.
- Preserved

In [2]:
# Imports and configurationimport json
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import chromadb
import torch

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

In [3]:
# Set up project paths and configuration:
PROJECT_ROOT = (
    Path.home()
    / "Schreibtisch"
    / "IfaDW"
    / "NLP"
    / "Final_Project"
    / "advance-ai-research-assistant"
)

CHUNKS_FILE = PROJECT_ROOT / "data" / "processed" / "document_chunks.jsonl"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma_db"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

SEMANTIC_TOP_K = 10
BM25_TOP_K = 10
RRF_TOP_K = 10
FINAL_TOP_K = 5

RRF_K = 60

print("Project:", PROJECT_ROOT)
print("Chunks file exists:", CHUNKS_FILE.exists())
print("ChromaDB exists:", CHROMA_DIR.exists())
print("CUDA available:", torch.cuda.is_available())

assert CHUNKS_FILE.exists()
assert CHROMA_DIR.exists()

Project: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant
Chunks file exists: True
ChromaDB exists: True
CUDA available: True


In [4]:
# Load document chunks
chunks = []

with CHUNKS_FILE.open("r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            chunks.append(json.loads(line))

chunks_by_id = {
    chunk["chunk_id"]: chunk
    for chunk in chunks
}

print("Chunks loaded:", len(chunks))
print("Unique chunk IDs:", len(chunks_by_id))

assert len(chunks) == len(chunks_by_id)

Chunks loaded: 156
Unique chunk IDs: 156
